In [1]:
#Brightway imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import bw2io as bi
import bw2calc as bc
import bw2data as bd
import bw2analyzer as ba
import os
import brightway2 as bw

14:54:36 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
#Temporalis imports
import bw_graph_tools as graph
from bw_temporalis import (
    easy_timedelta_distribution, 
    easy_datetime_distribution, 
    TemporalisLCA, 
    Timeline, 
    TemporalDistribution
)
from bw_temporalis.lcia import characterize_methane, characterize_co2

In [3]:
bw.projects.create_project("PharmaDLCA_2.5", biosphere="default", default_fp=False)
bw.projects.set_current("PharmaDLCA_2.5")
print("Current project:", bw.projects.current)

Current project: PharmaDLCA_2.5


In [5]:
bi.import_ecoinvent_release('3.10', 'cutoff', 'adarobinsonmedici', 'Ecoinvent37725921!')

Applying strategy: normalize_units
Applying strategy: drop_unspecified_subcategories
Applying strategy: ensure_categories_are_tuples
Applied 3 strategies in 0.01 seconds
Graph statistics for `ecoinvent-3.10-biosphere` importer:
4362 graph nodes:
	emission: 4000
	natural resource: 344
	inventory indicator: 13
	economic: 5
0 graph edges:
0 edges to the following databases:
0 unique unlinked edges (0 total):




100%|███████████████████████████████████████████████████████████████████████████| 4362/4362 [00:00<00:00, 23661.82it/s]


14:56:46 [info     ] Vacuuming database            
Created database: ecoinvent-3.10-biosphere
Extracting XML data from 23523 datasets
14:57:59 [info     ] Extracted 23523 datasets in 71.60 seconds
Applying strategy: normalize_units
Applying strategy: update_ecoinvent_locations
Applying strategy: remove_zero_amount_coproducts
Applying strategy: remove_zero_amount_inputs_with_no_activity
Applying strategy: remove_unnamed_parameters
Applying strategy: es2_assign_only_product_with_amount_as_reference_product
Applying strategy: assign_single_product_as_activity
Applying strategy: create_composite_code
Applying strategy: drop_unspecified_subcategories
Applying strategy: fix_ecoinvent_flows_pre35
Applying strategy: drop_temporary_outdated_biosphere_flows
Applying strategy: link_biosphere_by_flow_uuid
Applying strategy: link_internal_technosphere_by_composite_code
Applying strategy: delete_exchanges_missing_activity
Applying strategy: delete_ghost_exchanges
Applying strategy: remove_uncertain

100%|███████████████████████████████████████████████████████████████████████████| 23523/23523 [01:13<00:00, 319.45it/s]


14:59:23 [info     ] Vacuuming database            
Created database: ecoinvent-3.10-cutoff


In [6]:
bd.databases

Databases dictionary with 2 object(s):
	ecoinvent-3.10-biosphere
	ecoinvent-3.10-cutoff

In [8]:
DATA_DIR = r"C:\Users\arobinson\Desktop\DynamicLCA\Inventories"
infile = os.path.join(DATA_DIR, "lci_pharma_dynamic.csv") #infile will always have the same name
df = pd.read_csv(infile, usecols=[0,1,2,3,4])

# Rename columns
df.columns = [col.lower() for col in df.columns]
df = df.rename(columns={
    "time": "Time",
    "data_1": "CSTR",
    "data_2": "Hydrocyclone",
    "data_3": "Centrifuge",
    "data_4": "Pump",
})

# Compute 'Total'
df["Total"] = df["CSTR"] + df["Hydrocyclone"] + df["Centrifuge"] + df["Pump"]
df = df.iloc[:24]  # Only first 24 hours
df.head(2)

,Time,CSTR,Hydrocyclone,Centrifuge,Pump,Total
0,0,22.19214,2.19562,13.62883,5.86381,43.88040
1,1,22.19214,1.16313,13.62883,5.86381,42.84791


In [10]:
e_dk = pd.read_csv(r"C:\Users\arobinson\Desktop\DynamicLCA\Inventories\20250114_elect_dk.csv")
e_dk.head(1)

,Area,MTU,Biomass - Actual Aggregated [MW],Fossil Brown coal/Lignite - Actual Aggregated [MW],Fossil Coal-derived gas - Actual Aggregated [MW],Fossil Gas - Actual Aggregated [MW],Fossil Hard coal - Actual Aggregated [MW],Fossil Oil - Actual Aggregated [MW],Fossil Oil shale - Actual Aggregated [MW],Fossil Peat - Actual Aggregated [MW],...,Hydro Run-of-river and poundage - Actual Aggregated [MW],Hydro Water Reservoir - Actual Aggregated [MW],Marine - Actual Aggregated [MW],Nuclear - Actual Aggregated [MW],Other - Actual Aggregated [MW],Other renewable - Actual Aggregated [MW],Solar - Actual Aggregated [MW],Waste - Actual Aggregated [MW],Wind Offshore - Actual Aggregated [MW],Wind Onshore - Actual Aggregated [MW]
0,Denmark (DK),14.01.2025 00:00 - 14.01.2025 01:00 (CET/CEST),579,n/e,n/e,137,262,34,n/e,n/e,...,n/e,n/e,n/e,n/e,n/e,n/e,0,76,2223,3420


In [11]:
# Replace 'n/e' with 0 and convert to numeric BEFORE summing:
e_dk.iloc[:, 2:] = (
    e_dk.iloc[:, 2:]
    .replace("n/e", 0)
    .apply(pd.to_numeric, errors="coerce")
)

# Convert the time or hour into a simple integer index 
e_dk["MTU"] = range(len(e_dk)) 

# Select only numeric columns among the columns 2: onwards
cols_to_sum = e_dk.iloc[:, 2:].select_dtypes(include="number").columns 

# Now sum only these numeric columns
e_dk["Sum_elec_prod"] = e_dk[cols_to_sum].sum(axis=1)

e_dk.head(1)
#e_dk2.shape 
#e_dk2.info

C:\Users\arobinson\AppData\Local\Temp\ipykernel_22356\131688600.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace("n/e", 0)


,Area,MTU,Biomass - Actual Aggregated [MW],Fossil Brown coal/Lignite - Actual Aggregated [MW],Fossil Coal-derived gas - Actual Aggregated [MW],Fossil Gas - Actual Aggregated [MW],Fossil Hard coal - Actual Aggregated [MW],Fossil Oil - Actual Aggregated [MW],Fossil Oil shale - Actual Aggregated [MW],Fossil Peat - Actual Aggregated [MW],...,Hydro Water Reservoir - Actual Aggregated [MW],Marine - Actual Aggregated [MW],Nuclear - Actual Aggregated [MW],Other - Actual Aggregated [MW],Other renewable - Actual Aggregated [MW],Solar - Actual Aggregated [MW],Waste - Actual Aggregated [MW],Wind Offshore - Actual Aggregated [MW],Wind Onshore - Actual Aggregated [MW],Sum_elec_prod
0,Denmark (DK),0,579,0,0,137,262,34,0,0,...,0,0,0,0,0,0,76,2223,3420,6731


In [12]:
e_dk2 = e_dk.dropna(axis=1)

# This will be useful when uploading an yearly database. Slice the 24-hour rows (assuming exactly 24 rows), numeric columns only
df_day1 = e_dk.iloc[0:24, 2:-1]

# Calculate sums per column
sums_day1 = df_day1.sum(axis=1)

# Replace zeros in sums with NaN, to avoid division-by-zero
sums_day1 = sums_day1.replace({0: np.nan})

# Compute percentage of each cell relative to column total
percentage1 = df_day1.div(sums_day1, axis=0).fillna(0)

# Rename and add these percentage columns into e_dk
percentage1 = percentage1.add_suffix('_pct')  # e.g. "Biomass - Actual Aggregated [MW]_pct"
e_dk[percentage1.columns] = percentage1

C:\Users\arobinson\AppData\Local\Temp\ipykernel_22356\3745175061.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sums_day1 = sums_day1.replace({0: np.nan})
C:\Users\arobinson\AppData\Local\Temp\ipykernel_22356\3745175061.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  percentage1 = df_day1.div(sums_day1, axis=0).fillna(0)


In [13]:
percentage1["Sum_elec_prod_pct"] = (
    percentage1.drop(columns="Sum_elec_prod_pct", errors="ignore").sum(axis=1)
)
percentage1.head(1)

,Biomass - Actual Aggregated [MW]_pct,Fossil Brown coal/Lignite - Actual Aggregated [MW]_pct,Fossil Coal-derived gas - Actual Aggregated [MW]_pct,Fossil Gas - Actual Aggregated [MW]_pct,Fossil Hard coal - Actual Aggregated [MW]_pct,Fossil Oil - Actual Aggregated [MW]_pct,Fossil Oil shale - Actual Aggregated [MW]_pct,Fossil Peat - Actual Aggregated [MW]_pct,Geothermal - Actual Aggregated [MW]_pct,Hydro Pumped Storage - Actual Aggregated [MW]_pct,...,Hydro Water Reservoir - Actual Aggregated [MW]_pct,Marine - Actual Aggregated [MW]_pct,Nuclear - Actual Aggregated [MW]_pct,Other - Actual Aggregated [MW]_pct,Other renewable - Actual Aggregated [MW]_pct,Solar - Actual Aggregated [MW]_pct,Waste - Actual Aggregated [MW]_pct,Wind Offshore - Actual Aggregated [MW]_pct,Wind Onshore - Actual Aggregated [MW]_pct,Sum_elec_prod_pct
0,0.08602,0.0,0.0,0.020354,0.038924,0.005051,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.011291,0.330263,0.508097,1.0


In [33]:
cutoff_db = bd.Database("ecoinvent-3.10-cutoff")
filtered_acts = [
    act
    for act in cutoff_db
    if (
        ("electricity" in act["name"].lower() or "heat" in act["name"].lower())
        and "DK" in act["location"]  # Change to act["location"] == "DK" if you need an exact match
    )
]

print(f"Found {len(filtered_acts)} matching activities.\n")
for act in filtered_acts:
    print(f"- {act['name']} | Location: {act['location']} | Ref Product: {act.get('reference product')}")

Found 39 matching activities.

- electricity production, wind, 1-3MW turbine, offshore | Location: DK | Ref Product: electricity, high voltage
- electricity voltage transformation, residual mix, from medium to low voltage | Location: DK | Ref Product: electricity, low voltage
- heat and power co-generation, biogas, gas engine | Location: DK | Ref Product: electricity, high voltage
- electricity, medium voltage, residual mix | Location: DK | Ref Product: electricity, medium voltage
- heat and power co-generation, wood chips, 6667 kW, state-of-the-art 2014 | Location: DK | Ref Product: heat, district or industrial, other than natural gas
- heat and power co-generation, natural gas, conventional power plant, 100MW electrical | Location: DK | Ref Product: heat, district or industrial, natural gas
- electricity, from municipal waste incineration to generic market for electricity, medium voltage | Location: DK | Ref Product: electricity, medium voltage
- electricity voltage transformation fr

In [34]:
mapping = {
    # (Unchanged) Hard coal
    "Fossil Hard coal - Actual Aggregated [MW]_pct": 
        ("ecoinvent-3.10-cutoff", "heat and power co-generation, hard coal"),

    # (Unchanged) Oil
    "Fossil Oil - Actual Aggregated [MW]_pct": 
        ("ecoinvent-3.10-cutoff", "electricity production, oil"),

    # (CHANGED) Natural gas
    #   Old: "heat and power co-generation, natural gas, conventional power plant, 100MW electrical"
    #   New: "heat and power co-generation, natural gas, combined cycle power plant, 400MW electrical"
    "Fossil Gas - Actual Aggregated [MW]_pct": 
        ("ecoinvent-3.10-cutoff", "heat and power co-generation, natural gas, combined cycle power plant, 400MW electrical"),

    # (Unchanged) Wind offshore
    "Wind Offshore - Actual Aggregated [MW]_pct": 
        ("ecoinvent-3.10-cutoff", "electricity production, wind, 1-3MW turbine, offshore"),

    # (CHANGED) Wind onshore
    #   Old: "electricity production, wind, >3MW turbine, onshore"
    #   New: "electricity production, wind, 1-3MW turbine, onshore"
    "Wind Onshore - Actual Aggregated [MW]_pct": 
        ("ecoinvent-3.10-cutoff", "electricity production, wind, 1-3MW turbine, onshore"),

    # (Unchanged) Biomass
    "Biomass - Actual Aggregated [MW]_pct": 
        ("ecoinvent-3.10-cutoff", "heat and power co-generation, biogas, gas engine"),

    # (Unchanged) Waste
    "Waste - Actual Aggregated [MW]_pct": 
        ("ecoinvent-3.10-cutoff", "electricity, from municipal waste incineration to generic market for electricity, medium voltage"),

    # (CHANGED) Solar
    #   Old: "electricity production, photovoltaic, 3kWp slanted-roof installation, multi-Si, panel, mounted"
    #   New: "electricity production, photovoltaic, 3kWp slanted-roof installation, single-Si, panel, mounted"
    "Solar - Actual Aggregated [MW]_pct": 
        ("ecoinvent-3.10-cutoff", "electricity production, photovoltaic, 3kWp slanted-roof installation, single-Si, panel, mounted"),
}

In [36]:
#list(bd.methods)

In [29]:
matching_methods = [
    m for m in bd.methods
    if "ReCiPe" in str(m)
    and "endpoint (E)" in str(m)
    and "total" in str(m)
    and "ecosystem quality" in str(m)
]
print(f"Found {len(matching_methods)} matching methods.\n")
for method_tuple in matching_methods:
    print(method_tuple)

Found 2 matching methods.

('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (E) no LT', 'total: ecosystem quality no LT', 'ecosystem quality no LT')
('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (E)', 'total: ecosystem quality', 'ecosystem quality')


In [30]:
method = ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (E)', 'total: ecosystem quality', 'ecosystem quality')

assert method in bd.methods, f"Method {method} not found in bd.methods"
print("\nChosen LCIA method:", method)


Chosen LCIA method: ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (E)', 'total: ecosystem quality', 'ecosystem quality')


In [31]:
print("\nCheck that the method exists in bd.methods:")
assert method in bd.methods, f"Method {method} not found in bd.methods"
print("All good so far! You can now proceed to build a functional unit, do LCA, etc.")


Check that the method exists in bd.methods:
All good so far! You can now proceed to build a functional unit, do LCA, etc.


In [38]:
print("Current project:", bd.projects.current)
print("Databases:", list(bd.databases.keys()))
print("Methods in bd.methods (count):", len(bd.methods))
print("mapping dict:")
print("Using method:", method)
for k, v in mapping.items():
    print("  ", k, "=>", v)

Current project: PharmaDLCA_2.5
Databases: ['ecoinvent-3.10-biosphere', 'ecoinvent-3.10-cutoff']
Methods in bd.methods (count): 668
mapping dict:
Using method: ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (E)', 'total: ecosystem quality', 'ecosystem quality')
   Fossil Hard coal - Actual Aggregated [MW]_pct => ('ecoinvent-3.10-cutoff', 'heat and power co-generation, hard coal')
   Fossil Oil - Actual Aggregated [MW]_pct => ('ecoinvent-3.10-cutoff', 'electricity production, oil')
   Fossil Gas - Actual Aggregated [MW]_pct => ('ecoinvent-3.10-cutoff', 'heat and power co-generation, natural gas, combined cycle power plant, 400MW electrical')
   Wind Offshore - Actual Aggregated [MW]_pct => ('ecoinvent-3.10-cutoff', 'electricity production, wind, 1-3MW turbine, offshore')
   Wind Onshore - Actual Aggregated [MW]_pct => ('ecoinvent-3.10-cutoff', 'electricity production, wind, 1-3MW turbine, onshore')
   Biomass - Actual Aggregated [MW]_pct => ('ecoinvent-3.10-cutoff', 'heat and power co-

In [41]:
#Define the Debug Function

def compute_lca_score_debug(equipment, hour_index):
    """
    Build a functional unit for 'equipment' at 'hour_index',
    using fraction of each electricity technology from `percentage1`,
    then run a single LCA. Returns the LCA score (float).

    Includes step-by-step debug prints. If anything fails
    (missing data, method mismatch, etc.), raises an error
    with a clear message.
    """
    print(f"\n[DEBUG] Starting LCA calculation for equipment='{equipment}', hour={hour_index}")

    # --- STEP A) Basic checks on input ---
    # 1) Check that hour_index is valid in df
    if hour_index not in df.index:
        raise ValueError(
            f"[ERROR] hour_index={hour_index} is not in df.index. "
            f"Available indices: {df.index.tolist()}"
        )

    # 2) Check that 'equipment' is a valid column
    if equipment not in df.columns:
        raise ValueError(
            f"[ERROR] equipment='{equipment}' not found in df.columns: {df.columns.tolist()}"
        )

    # 3) Get consumption
    consumption = df.loc[hour_index, equipment]
    if not pd.api.types.is_number(consumption):
        raise TypeError(
            f"[ERROR] df[{hour_index}, {equipment}] is not numeric: {consumption}"
        )
    print(f"[DEBUG] consumption={consumption} (hour={hour_index}, equipment={equipment})")

    # --- STEP B) Build the FU ---
    fu = {}

    for tech_col, (db_name, act_name) in mapping.items():
        # Check if tech_col is in percentage1
        if tech_col not in percentage1.columns:
            print(f"[WARNING] tech_col='{tech_col}' not found in percentage1.columns. Skipping.")
            continue

        fraction = percentage1.loc[hour_index, tech_col]
        if fraction == 0:
            continue

        partial_amount = consumption * fraction
        print(
            f"[DEBUG] For tech_col='{tech_col}', fraction={fraction} => partial_amount={partial_amount}"
        )

        # Check db_name is in bd.databases
        if db_name not in bd.databases:
            raise ValueError(
                f"[ERROR] db_name='{db_name}' not found in bd.databases: {list(bd.databases.keys())}"
            )

        # Search for the activity by name
        the_db = bd.Database(db_name)
        matches = [act for act in the_db if act["name"] == act_name]

        if not matches:
            raise ValueError(f"[ERROR] No activity named '{act_name}' found in DB='{db_name}'")
        if len(matches) > 1:
            print(f"[WARNING] Multiple matches for '{act_name}' in DB='{db_name}'. Using first match.")

        node = matches[0]
        fu[node.key] = partial_amount

    if not fu:
        print(f"[DEBUG] Functional unit is empty. Returning 0.0 for equipment='{equipment}', hour={hour_index}")
        return 0.0

    print(f"[DEBUG] Built FU with {len(fu)} items => {fu}")

    # --- STEP C) Prepare LCA Inputs ---
    if method not in bd.methods:
        raise ValueError(
            f"[ERROR] method={method} not found in bd.methods.\n"
            f"Possible ReCiPe methods => {[m for m in bd.methods if 'ReCiPe' in str(m)]}"
        )

    try:
        fu_prepared, data_objs, remapping = bd.prepare_lca_inputs(
            demand=fu,
            method=method
        )
    except Exception as e:
        raise RuntimeError(f"[ERROR] prepare_lca_inputs failed: {e}")

    print(f"[DEBUG] prepare_lca_inputs => fu_prepared={fu_prepared}, data_objs={data_objs}")

    # --- STEP D) Build & run LCA ---
    try:
        lca = bc.LCA(demand=fu_prepared, data_objs=data_objs)
        lca.lci()
        lca.lcia()
    except Exception as e:
        raise RuntimeError(f"[ERROR] LCA calculation failed: {e}")

    score = lca.score
    print(f"[DEBUG] Final LCA score={score} for equipment='{equipment}', hour={hour_index}")
    return score


print("Function 'compute_lca_score_debug' defined successfully!")

Function 'compute_lca_score_debug' defined successfully!


In [42]:
# Calculate LCA Scores For Each Hour & Equipment
equipments = ["CSTR", "Hydrocyclone", "Centrifuge", "Pump", "Total"]
results_df = pd.DataFrame(index=range(24), columns=equipments)

for eqp in equipments:
    for h in range(24):
        score = compute_lca_score_debug(eqp, h)
        results_df.loc[h, eqp] = score

print("\n=== Final Results DataFrame ===")
display(results_df)


[DEBUG] Starting LCA calculation for equipment='CSTR', hour=0
[DEBUG] consumption=22.19214 (hour=0, equipment=CSTR)
[DEBUG] For tech_col='Fossil Hard coal - Actual Aggregated [MW]_pct', fraction=0.03892437973555193 => partial_amount=0.8638152845045313
[WARNING] Multiple matches for 'heat and power co-generation, hard coal' in DB='ecoinvent-3.10-cutoff'. Using first match.
[DEBUG] For tech_col='Fossil Oil - Actual Aggregated [MW]_pct', fraction=0.005051255385529639 => partial_amount=0.1120981666914277
[WARNING] Multiple matches for 'electricity production, oil' in DB='ecoinvent-3.10-cutoff'. Using first match.
[DEBUG] For tech_col='Fossil Gas - Actual Aggregated [MW]_pct', fraction=0.020353587876987076 => partial_amount=0.45168967166839996
[WARNING] Multiple matches for 'heat and power co-generation, natural gas, combined cycle power plant, 400MW electrical' in DB='ecoinvent-3.10-cutoff'. Using first match.
[DEBUG] For tech_col='Wind Offshore - Actual Aggregated [MW]_pct', fraction=0.3

,CSTR,Hydrocyclone,Centrifuge,Pump,Total
0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0


In [43]:
# Tell Pandas to use scientific notation with e.g. 10 decimal places:
pd.set_option('display.float_format', '{:.10e}'.format)

# Now show the DataFrame again
display(results_df)

,CSTR,Hydrocyclone,Centrifuge,Pump,Total
0,9.2809879181e-08,6.7580713964e-09,5.1589666811e-08,3.2921623221e-08,1.1600213888e-07
1,8.9665011249e-08,3.9603517900e-09,5.5734992067e-08,1.5510362732e-08,2.5310144781e-07
2,5.6083162706e-08,3.5865268372e-09,3.8843711223e-08,1.4846309110e-08,1.7888050258e-07
3,3.5585827166e-08,3.9615608569e-09,7.7850087262e-08,3.2993291193e-08,2.0801753892e-07
4,1.2381766167e-07,4.1887387254e-09,3.5882975002e-08,2.6061171557e-08,2.0043889659e-07
5,9.2813886568e-08,9.4665520681e-09,3.3385875276e-08,2.5199768578e-08,7.3774400994e-08
6,6.0196509470e-08,4.4454735772e-09,6.2803187319e-08,1.2889203248e-08,1.1105186440e-07
7,8.0199940891e-08,3.9648071744e-09,3.1564576931e-08,2.4513311798e-08,2.1736278279e-07
8,5.7151989374e-08,6.0873656188e-09,3.8021872350e-08,1.3118427635e-08,1.9913808325e-07
9,7.9921219178e-08,4.8714455467e-09,3.8411617247e-08,2.7036053232e-08,1.7255016440e-07


In [59]:
db_name = "KTB1_dynamic"
if db_name in bd.databases:
    del bd.databases[db_name]

db_dyn = bw.Database(db_name)
db_dyn.register()

# We'll create exactly ONE dataset:
dataset_key = (db_name, "KTB1 dynamic consumption")
ktb1_dataset = {
    dataset_key: {
        'name': "KTB1 dynamic consumption (24h)",
        'unit': 'megawatt hour',
        'type': 'process',
        'exchanges': []
    }
}

In [61]:
exchanges_list = []
time_hours = np.array(df.index, dtype='timedelta64[h]')  # shape = (24,)

for eqp in equipments:
    for tech_col, (db_name_map, act_name) in mapping.items():
        consumption_array = []
        for h in range(24):
            fraction = percentage1.loc[h, tech_col]
            consumption_hour = df.loc[h, eqp]  # in MW
            partial = consumption_hour * fraction
            consumption_array.append(partial)

        # If the entire array is zeros, skip
        if all(v == 0 for v in consumption_array):
            continue

        # Build the temporal distribution
        dist = TemporalDistribution(
            date=time_hours,
            amount=np.array(consumption_array)
        )

        # Find the ecoinvent activity
        db_acts = [act for act in bd.Database(db_name_map) if act["name"] == act_name]
        if not db_acts:
            print(f"[WARNING] Activity '{act_name}' not found in '{db_name_map}'. Skipping.")
            continue
        # pick the first match
        tech_act = db_acts[0]

        # Add exchange
        exg = {
            'input': tech_act.key,
            'amount': dist.total,  # sum of consumption_array
            'type': 'technosphere',
            'TemporalDistribution': dist,
            'unit': 'megawatt hour',
            # flow name: e.g. "CSTR via Fossil Hard coal"
            'name': f"{eqp} via {tech_col}"
        }
        exchanges_list.append(exg)

ktb1_dataset[dataset_key]['exchanges'] = exchanges_list
db_dyn.write(ktb1_dataset)

print(f"{len(exchanges_list)} exchanges created in '{db_name}' dataset.")


18:15:41 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<?, ?it/s]


18:15:41 [info     ] Vacuuming database            
40 exchanges created in 'KTB1_dynamic' dataset.


In [63]:
fu_dyn = {(db_name, "KTB1 dynamic consumption"): 1}

# Traditional LCA (aggregated)
lca = bc.LCA(fu_dyn, method=method)
lca.lci()
lca.lcia()
print(f"\nAggregated LCA result for 24h total: {lca.score} (method={method})")


Aggregated LCA result for 24h total: 7.786300402708478e-06 (method=('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (E)', 'total: ecosystem quality', 'ecosystem quality'))


In [64]:
# Temporalis LCA
lca_temporal = TemporalisLCA(lca)
timeline = lca_temporal.build_timeline()
df_timeline = timeline.build_dataframe()

print("\nTimeline DataFrame (first 10 rows):")
display(df_timeline.head(10))
print(df_timeline.columns)

Starting graph traversal
Calculation count: 1158


MultipleTechnosphereExchanges: Found 2 exchanges for link between (ecoinvent-3.10-cutoff|4fd1e0a42541e761f545b18d2c595358|heat and power co-generation, hard coal) and (KTB1_dynamic|KTB1 dynamic consumption|KTB1 dynamic consumption (24h))